In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [4]:
!pip install mlflow
import mlflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 70.6 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 81.7 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 50.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 879.5/879.5 kB 36.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 10.6 MB/s eta 0:00:00


In [6]:
!pip install dagshub -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 5.7 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 5.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ydata-profiling 4.18.1 requires dacite<2,>=1.9, but you have dacite 1.6.0 which is incompatible.


In [19]:
import sys
sys.path.append('/kaggle/usr/lib/notebooks/nikolozdodashvili/preprocessing/notebooks/nikolozdodashvili')

from preprocessing import (
    optimize_memory,
    DropHighMissingFeatures,
    FrequencyEncoder,
    MissingValueImputer,
    TimeFeatureExtractor,
    TransactionAmtTransformer,
    GroupAggregator,
    DropCorrelatedFeatures
)

In [23]:
import pandas as pd
import mlflow.sklearn
import gc
import numpy as np


from preprocessing import optimize_memory as reduce_mem_usage


DAGSHUB_MLFLOW_URI = "https://dagshub.com/ndoda23/MachineLearning---IEEE-CIS-Fraud-Detection.mlflow"
mlflow.set_tracking_uri(DAGSHUB_MLFLOW_URI)

model_name = "LightGBM_Best_model"
model_version = 1 
model_uri = f"models:/{model_name}/{model_version}"

loaded_pipeline = mlflow.sklearn.load_model(model_uri)

test_trans = reduce_mem_usage(pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv'))
test_id = reduce_mem_usage(pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv'))

print(f"Memory optimized! Shapes: Trans {test_trans.shape}, ID {test_id.shape}")

test_raw = pd.merge(test_trans, test_id, on='TransactionID', how='left')

del test_trans, test_id
gc.collect()

fix_columns = {col: col.replace('-', '_') for col in test_raw.columns if '-' in col}
test_raw = test_raw.rename(columns=fix_columns)


X_test_input = test_raw.drop(columns=['TransactionID'])

test_probs = loaded_pipeline.predict_proba(X_test_input)[:, 1]

submission = pd.DataFrame({
    'TransactionID': test_raw['TransactionID'],
    'isFraud': test_probs
})

submission.to_csv('submission.csv', index=False)


მეხსიერება დამუშავებამდე: 1519.24 MB
მეხსიერება დამუშავების შემდეგ: 738.36 MB
შემცირდა: 51.4%
მეხსიერება დამუშავებამდე: 44.39 MB
მეხსიერება დამუშავების შემდეგ: 15.80 MB
შემცირდა: 64.4%
Memory optimized! Shapes: Trans (506691, 393), ID (141907, 41)
